Chain Pricing for Milk Products: Genevieve Silver

DellaVigna and Gentzkow (2019) find that many US grocery stores price at the chain level.
My code helps understand the variation in chain-level pricing for milk products.
Essentially, I run price regressions using different levels of fixed effects including store-level and chain-level and see how much of the variation in the data is explained by those fixed effects. 

Setup

In [ ]:
!pip3 install statsmodels
!pip3 install linearmodels

In [273]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

Load Data

In [274]:
# Adjust path for file location
# These are the Nielsen files of milk scanner data you need for input, I used the 2018 data
PRODUCTS_FILE_PATH = r'/Users/gigisilver/Desktop/products.tsv' 
SCANNER1_FILE_PATH = r'/Users/gigisilver/Desktop/3592_2018.tsv'
STORES_FILE_PATH = r'/Users/gigisilver/Desktop/storesScanner_2018.tsv'

# Load Data
products = pd.read_csv (PRODUCTS_FILE_PATH, sep='\t', encoding='latin1')
# Add additional scanner movement files for the year if computer has enough RAM
scanner1 = pd.read_csv (SCANNER1_FILE_PATH, sep='\t', encoding='latin1') 
stores = pd.read_csv (STORES_FILE_PATH, sep='\t', encoding='latin1')

# Sample data if needed
SAMPLE_RATE = 0.2
stores = stores.sample(frac=SAMPLE_RATE, replace=True, random_state=1)
scanner1 = scanner1.sample(frac=SAMPLE_RATE, replace=True, random_state=1)

Create Merged Data Set

In [275]:
# Join the products data with the scanner1 data on upc
products_scanner1_df = pd.merge(products, scanner1, how='inner', on = 'upc')

# Create a new data frame which restricts data set to only milk products
# by only including rows where the product group code = the product group description of "MILK".
onlymilk_purchases_df = products_scanner1_df.loc[products_scanner1_df['product_group_code'].isin([2506, 1012])]

# Join the milk purchases data with the stores data on store_code_uc.
final_df = pd.merge(onlymilk_purchases_df, stores, how='inner', on = 'store_code_uc')

Create Regression for Store-Level Fixed Effects

In [276]:
# Only set index if we haven't already
if not 'week_end' in final_df.index.names:
   final_df["week_end"] = pd.to_datetime(final_df.week_end.astype(str))
   final_df = final_df.set_index(["upc", "week_end"])

In [277]:
from linearmodels.panel import PanelOLS

exog_vars = ["store_code_uc"]
exog = sm.add_constant(final_df[exog_vars])
mod = PanelOLS(final_df.price, exog, entity_effects=True)
fe_res = mod.fit()
print(fe_res)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  price   R-squared:                        0.0004
Estimator:                   PanelOLS   R-squared (Between):             -0.0005
No. Observations:              581305   R-squared (Within):               0.0004
Date:                Sat, Sep 24 2022   R-squared (Overall):           9.955e-05
Time:                        15:56:55   Log-likelihood                -3.111e+05
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      257.13
Entities:                         772   P-value                           0.0000
Avg Obs:                       752.99   Distribution:                F(1,580532)
Min Obs:                       1.0000                                           
Max Obs:                    2.679e+04   F-statistic (robust):             257.13
                            

Find Chains
- Valid chains are those in which at least 80% of stores with that retailer_code have the same parent_code 
according to "Uniform Pricing in U.S. Retail Chains" by DellaVigna and Gentzkow (2019)

In [278]:
# Group stores dataset by retailer_code and then parent_code
# Count number of uniqe retailer_code parent_code combonations and create new column "store_count" with this count
combo_count_df = stores.groupby(['retailer_code','parent_code'])['store_code_uc'].count().reset_index(name="store_count")
#pd.set_option('display.max_rows', 10)
#combo_count

In [279]:
# Create new column "parent_total" 
# Group store_count by parent_code and find the total number of the same parent_code for each retaoler code
combo_count_df['parent_total'] = combo_count_df.groupby(['parent_code'])['store_count'].transform('sum').reset_index(name="parent_total")['parent_total']

In [280]:
# Create new column = to the store_count/parent_total
combo_count_df["same_parent_percent"] = combo_count_df["store_count"] / combo_count_df["parent_total"]

# Create a new column "is_chain" which returns true if the "same_parent_percent" is greater than 80%
# This will allow us to know if a retialer is apart of a chain by definition of the Uniform Pricing in U.S. Retail Chains Paper  
combo_count_df["is_chain"] = combo_count_df['same_parent_percent'] > 0.8
#combo_count.loc[combo_count['same_parent_percent'] > 0.8]

In [281]:
# Set is_chain to a 0 or 1 instead of true or false 
combo_count_df['is_chain'] = np.where(combo_count_df['is_chain'], 1,  0)

In [282]:
#Create a new collunm chain_id and set it to the parent code of the chain if it is a chain , else set it to 0
combo_count_df['chain_id'] = np.where(combo_count_df['is_chain'] == 1, combo_count_df["parent_code"], 0)

Create Regression for Chain-Level Fixed Effects

In [283]:
# Reset index on final_df so that purchase_date is included in chain_df
final_df = final_df.reset_index()

# Merge combo_count_df with final_df before running regression
chain_df = pd.merge(combo_count_df, final_df, how='inner', on = 'retailer_code')

In [285]:
# Create new collumn with same values in "week_end" to use as exog variable, because we use week_end as panel index
chain_df["week_end_2"] = chain_df["week_end"]

In [286]:
# Factorize upc number and week_end number in new columns (reggression can't handle large numbers and will error otherwise)
chain_df['upc_2']=pd.factorize(chain_df['upc'].tolist())[0]
chain_df['week_end_2']=pd.factorize(chain_df['week_end_2'].tolist())[0]

In [287]:
# Set index for Panel Regression
if not 'week_end' in chain_df.index.names:
    chain_df["week_end"] = pd.to_datetime(chain_df.week_end.astype(str))
    chain_df = chain_df.set_index(["upc_2", "week_end"])

In [288]:
chain_sf_dummies = pd.get_dummies(data = chain_df, prefix="we", columns = ['week_end_2'], drop_first = True)

In [289]:
weekend_dummy_cols = [col for col in chain_sf_dummies.columns if 'we_' in col]

In [296]:
from linearmodels.panel import PanelOLS

exog_vars = ["size1_code_uc",'units','department_code','brand_code_uc','store_zip3','retailer_code',"chain_id",'product_module_code'] + weekend_dummy_cols
exog = sm.add_constant(chain_sf_dummies[exog_vars])
mod = PanelOLS(chain_sf_dummies.price, exog)
fe_res = mod.fit()
print(fe_res)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  price   R-squared:                        0.5374
Estimator:                   PanelOLS   R-squared (Between):              0.5129
No. Observations:              647889   R-squared (Within):              -0.2781
Date:                Sat, Sep 24 2022   R-squared (Overall):              0.5374
Time:                        15:59:08   Log-likelihood                -8.264e+05
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                   1.275e+04
Entities:                         772   P-value                           0.0000
Avg Obs:                       839.23   Distribution:               F(59,647829)
Min Obs:                       1.0000                                           
Max Obs:                     2.97e+04   F-statistic (robust):          1.275e+04
                            